<a href="https://colab.research.google.com/github/dominicwooldridge/GSB-S544/blob/Practice-Activities/Practice_Activity_4_1_Webscraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XML, HTML, and Web Scraping

JSON and XML are two different ways to represent hierarchical data. Which one is better? There are lots of articles online which discuss similarities and differences between JSON and XML and their advantages and disadvantages. Both formats are still in current usage, so it is good to be familiar with both. However, JSON is more common, so we'll focus on working with JSON representations of hierarchical data.

The reading covered an example of using Beautiful Soup to parse XML. Rather than doing another example XML now, we'll skip straight to scraping HTML from a webpage. Both HTML and XML can be parsed in a similar way with Beautiful Soup.

In [53]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re


## Scraping an HTML table with Beautiful Soup

Open the URL https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population and scroll down until you see a table of the cities in the U.S. with population over 100,000 (as of Jul 1, 2022). We'll use Beautiful Soup to scrape information from this table.

*Read* in the HTML from the ULR using the `requests` library.

In [37]:
# YOUR CODE HERE
import requests
url = "https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population"
response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
print(response.status_code)
html = response.text
print(html[:500])

200
<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 vector-feature-night-mode-enabled skin-theme-clientpref-day vect


Use Beautiful Soup to parse this string into a tree called `soup`

In [38]:
# YOUR CODE HERE
from bs4 import BeautifulSoup
soup = BeautifulSoup(html, "html.parser")
print(soup.title.get_text(strip=True))

List of United States cities by population - Wikipedia


To find an HTML tag corresponding to a specific element on a webpage, right-click on it and choose "Inspect element". Go to the cities table Wikipedia page and do this now.

You should find that the cities table on the Wikipedia page corresponds to the element

```
<table class="wikitable sortable jquery-tablesorter" style="text-align:center">
```

There are many `<table>` tags on the page.

In [39]:
len(soup.find_all("table"))

10

We can use attributes like `class=` and `style=` to narrow down the list.

In [40]:
len(soup.find_all("table",
                  attrs={
                      "class": "wikitable sortable",
                      "style": "text-align:center"}
                  ))

0

At this point, you can manually inspect the tables on the webpage to find that the one we want is the first one (see `[0]` below). We'll store this as `table`.

In [41]:
tables = soup.select("table.wikitable.sortable")
print("found:", len(tables))

if not tables:
    raise RuntimeError("No sortable wikitable found.")


found: 4


**Now you will write code to scrape the information in `table` to create a Pandas data frame with one row for each city and columns for: city, state, population (2022 estimate), and 2020 land area (sq mi).** Refer to the Notes/suggestions below as you write your code. A few Hints are provided further down, but try coding first before looking at the hints.

Notes/suggestions:

- Use as a guide the code from the reading that produced the data frame of Statistics faculty
- Inspect the page source as you write your code
- You will need to write a loop to get the information for all cities, but you might want to try just scraping the info for New York first
- You will need to pull the text from the tag. If `.text` returns text with "\n" at the end, try `.get_text(strip = True)` instead of `.text`
- Don't forget to convert to a Pandas Data Frame; it should have 333 rows and 4 columns
- The goal of this exercise is just to create the Data Frame. If you were going to use it --- e.g., what is the population density for all cities in CA? --- then you would need to clean the data first (to clean strings and convert to quantitative). (You can use Beautiful Soup to do some of the cleaning for you, but that goes beyond our scope.)

In [47]:
# YOUR CODE HERE. ADD AS MANY CELLS AS NEEDED

URL = "https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population"

resp = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"})
resp.raise_for_status()
html = resp.text


soup = BeautifulSoup(html, "html.parser")


tables = soup.select("table.wikitable.sortable")
if not tables:
    raise RuntimeError("No sortable wikitable found on the page.")
table = next((t for t in tables if t.caption and "100,000" in t.caption.get_text()),
             tables[0])


def clean_text(s: str) -> str:
    if s is None:
        return ""

    s = re.sub(r"\s*\[[^\]]*\]\s*", "", s)
    return s.strip()

thead = table.find("thead")
if thead and thead.find_all("th"):
    header_cells = thead.find_all("th")
else:
    first_tr = table.find("tr")
    header_cells = first_tr.find_all(["th", "td"])

headers = [clean_text(th.get_text(" ", strip=True)).lower() for th in header_cells]



def find_col(candidates):
    for i, h in enumerate(headers):
        if any(c in h for c in candidates):
            return i
    return None

ix_city   = find_col(["city"])
ix_state  = find_col(["state", "st"])
ix_pop    = find_col(["2024 estimate","2023 estimate","2022 estimate","2022 population","2022 est"])
ix_land20 = find_col(["2020 land area","2020 land"])

needed = [ix_city, ix_state, ix_pop, ix_land20]
if not all(i is not None for i in needed):
    raise RuntimeError(f"Missing required columns. Headers found: {headers}")


if thead:
    data_trs = table.find("tbody").find_all("tr")
else:
    all_trs = table.find_all("tr")
    data_trs = all_trs[1:]

rows = []
for tr in data_trs:

    if "class" in tr.attrs and "sortbottom" in tr["class"]:
        continue
    tds = tr.find_all(["td", "th"])
    if len(tds) <= max(needed):
        continue
    city   = clean_text(tds[ix_city].get_text(strip=True))
    state  = clean_text(tds[ix_state].get_text(strip=True))
    pop    = clean_text(tds[ix_pop].get_text(strip=True))
    land20 = clean_text(tds[ix_land20].get_text(strip=True))
    if city and state:
        rows.append({
            "city": city,
            "state": state,
            "population_2022_est": pop,
            "land_area_2020_sq_mi": land20
        })


df = pd.DataFrame(rows, columns=["city","state","population_2022_est","land_area_2020_sq_mi"])


print(df.shape)
df.head()

(346, 4)


,city,state,population_2022_est,land_area_2020_sq_mi
0,New York,NY,"8,478,072",300.5
1,Los Angeles,CA,"3,878,704",469.5
2,Chicago,IL,"2,721,308",227.7
3,Houston,TX,"2,390,125",640.4
4,Phoenix,AZ,"1,673,164",518.0


Hints:

- Each city is a row in the table; find all the `<tr>` tags to find all the cities
- Look for the `<td>` tag to see table entries within a row
- The rank column is represented by `<th>` tags, rather than `<td>` tags. So within a row, the first (that is, `[0]`) `<td>` tag corresponds to the city name.

## Aside: Scraping an HTML table with Pandas



The Pandas command `read_html` can be used to scrape information from an HTML table on a webpage.

We can call `read_html` on the URL.

In [52]:
pd.read_html("https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population")

HTTPError: HTTP Error 403: Forbidden

However, this scrapes all the tables on the webpage, not just the one we want. As with Beautiful Soup, we can narrow the search by specifying the table attributes.

In [49]:
pd.read_html("https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population", attrs = {'class': 'wikitable sortable', "style": "text-align:center"})

HTTPError: HTTP Error 403: Forbidden

This still returns 3 tables. As we remarked above, the table that we want is the first one (see `[0]` below).

In [51]:
df_cities2 = pd.read_html("https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population", attrs = {'class': 'wikitable sortable', "style": "text-align:center"})[0]
df_cities2

HTTPError: HTTP Error 403: Forbidden

Wait, that seemed much easier than using Beautiful Soup, and it returned a data frame, and we even got for free some formatting like removing the commas from the population! Why didn't we just use `read_html` in the first place? It's true the `read_html` works well when scraping information from an HTML *table*. Unfortunately, you often want to scrape information from a webpage that isn't conveniently stored in an HTML table, in which case `read_html` won't work. (It only searches for `<table>`, `<th>`, `<tr>`, and `<td>` tags, but there are many other HTML tags.) Though Beautiful Soup is not as simple as `read_html`, it is more flexible and thus more widely applicable.

## Scraping information that is NOT in a `<table>` with Beautiful Soup

The Cal Poly course catalog http://catalog.calpoly.edu/collegesandprograms/collegeofsciencemathematics/statistics/#courseinventory contains a list of courses offered by the Statistics department. **You will scrape this website to obtain a Pandas data frame with one row for each DATA or STAT course and two columns: course name and number (e.g, DATA 301. Introduction to Data Science) and term typically offered (e.g., Term Typically Offered: F, W, SP).**

Note: Pandas `read_html` is not help here since the courses are not stored in a `<table>.`

In [54]:
pd.read_html("http://catalog.calpoly.edu/collegesandprograms/collegeofsciencemathematics/statistics/#courseinventory")

[                                        Program name   Program type
 0                              Actuarial Preparation          Minor
 1  Cross Disciplinary Studies Minor in Bioinforma...          Minor
 2   Cross Disciplinary Studies Minor in Data Science          Minor
 3                                         Statistics  BS, MS, Minor]


Notes/suggestions:


- Inspect the page source as you write your code
- The courses are not stored in a `<table>`. How are they stored?
- You will need to write a loop to get the information for all courses, but you might want to try just scraping the info for DATA 100 first
- What kind of tag is the course name stored in? What is the `class` of the tag?
- What kind of tag is the quarter(s) the course is offered stored in? What is the `class` of the tag? Is this the only tag of this type with the class? How will you get the one you want?
- You don't have to remove the number of units (e.g., 4 units) from the course name and number, but you can try it if you want
- You will need to pull the text from the tag. If `.text` returns text with "\n" at the end, try `get_text(strip = True)` instead of `text`
- Don't forget to convert to a Pandas Data Frame; it should have 74 rows and 2 columns
- The goal of this exercise is just to create the Data Frame. If you were going to use it then you might need to clean the data first. (You can use Beautiful Soup to do some of the cleaning for you, but that goes beyond our scope.)



In [ ]:
# YOUR CODE HERE. ADD AS MANY CELLS AS NEEDED

In [56]:


url = "http://catalog.calpoly.edu/collegesandprograms/collegeofsciencemathematics/statistics/#courseinventory"


resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
resp.raise_for_status()
html = resp.text


soup = BeautifulSoup(html, "html.parser")


blocks = soup.find_all("div", {"class": "courseblock"})
print("course blocks found:", len(blocks))  # sanity check

rows = []
for b in blocks:

    title_p = b.find("p", {"class": "courseblocktitle"})
    if not title_p:
        continue


    for sp in title_p.find_all("span"):
        sp.extract()
    course = title_p.get_text(" ", strip=True)

    course = re.sub(r"\s*\.\s*$", "", course)


    offered = ""
    for p in b.find_all("p", {"class": "noindent"}):
        txt = p.get_text(" ", strip=True)
        m = re.search(r"(Typically\s*Offered|Term\s*Typically\s*Offered)\s*:\s*(.+)", txt, flags=re.I)
        if m:
            offered = m.group(2)
            break

    rows.append({"course": course, "term_typically_offered": offered})


df = pd.DataFrame(rows, columns=["course", "term_typically_offered"])

print(df.shape)
df.head()


course blocks found: 74
(74, 2)


,course,term_typically_offered
0,DATA 100. Data Science for All I,"F, W, SP"
1,DATA 301. Introduction to Data Science,"F, W, SP"
2,DATA 401. Data Science Process and Ethics,F
3,DATA 402. Mathematical Foundations of Data Sci...,F
4,DATA 403. Data Science Projects Laboratory,F


Hints:

- Each course is represented by a `<div>` with `class=courseblock`, so you can find all the courses with `soup.find_all("div", {"class": "courseblock"})`
- The course name is in a `<p>` tag with `class=courseblocktitle`, inside a `<strong>` tag. (Though I don't think we need to find the strong tag here.)
- The term typically offered is in `<p>` tag with `class=noindent`. However, there are several tags with this class; term typically offered is the first one.
- If you want to use Beautiful Soup to remove the course units (e.g., 4 units), find the `<span>` tag within the course name tag and `.extract()` this span tag